# Clustering Model Training


This notebook trains clustering models to identify demand patterns in daily electricity consumption profiles.

## Models Used:
- **StandardScaler**: Standardizes 96-point daily profile features
- **K-Means**: Groups similar daily demand profiles into clusters
- **DBSCAN**: Density-based clustering for pattern detection
- **Silhouette Score**: Evaluates clustering quality
- **Davies-Bouldin Index**: Another clustering quality metric

## Steps:
1. Load daily profiles from data/processed/daily_profiles.csv
2. Select p00 to p95 features for clustering
3. Train StandardScaler and save to models/profile_scaler.pkl
4. Test K-Means with K=2 to 10 and select best K
5. Train final K-Means and save to models/kmeans_model.pkl
6. Train DBSCAN and save to models/dbscan_model.pkl
7. Create clustering results and save metrics


In [ ]:

import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Import sklearn components
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Set up paths
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
DAILY_PROFILES_PATH = DATA_PROCESSED_DIR / "daily_profiles.csv"
CLUSTERING_RESULTS_PATH = DATA_PROCESSED_DIR / "clustering_results.csv"

# Create models directory
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Daily profiles path: {DAILY_PROFILES_PATH}")
print(f"Models directory: {MODELS_DIR}")


In [ ]:

# Load daily profiles
if not DAILY_PROFILES_PATH.exists():
    raise FileNotFoundError(f"Daily profiles not found at {DAILY_PROFILES_PATH}")

daily_profiles = pd.read_csv(DAILY_PROFILES_PATH)

print(f"Daily profiles loaded successfully!")
print(f"Shape: {daily_profiles.shape}")
print(f"Date range: {daily_profiles['date'].min()} to {daily_profiles['date'].max()}")

# Get profile columns
profile_cols = [f"p{str(i).zfill(2)}" for i in range(96)]
available_cols = [col for col in profile_cols if col in daily_profiles.columns]

print(f"Profile columns available: {len(available_cols)}")
print(f"\nFirst few rows:")
print(daily_profiles[['date', 'daily_mean', 'daily_peak']].head())


In [ ]:

# Prepare features for clustering
X = daily_profiles[available_cols].values

print(f"Features prepared for clustering:")
print(f"- Number of samples (days): {X.shape[0]}")
print(f"- Number of features (time slots): {X.shape[1]}")
print(f"- Feature shape: {X.shape}")

# Check for missing values
missing_values = np.isnan(X).sum()
print(f"- Missing values: {missing_values}")

if missing_values > 0:
    print("Warning: Missing values found. Filling with mean values.")
    from sklearn.impute import SimpleImputer
    imputer = SimpleImputer(strategy='mean')
    X = imputer.fit_transform(X)
    print(f"Missing values after imputation: {np.isnan(X).sum()}")


In [ ]:

# Train StandardScaler
print("Training StandardScaler...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("StandardScaler trained successfully!")
print(f"Scaled data shape: {X_scaled.shape}")
print(f"Scaled data mean: {X_scaled.mean():.6f} (should be ~0)")
print(f"Scaled data std: {X_scaled.std():.6f} (should be ~1)")

# Save scaler
with open(MODELS_DIR / "profile_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print(f"Scaler saved to: {MODELS_DIR / 'profile_scaler.pkl'}")


In [ ]:

# Find optimal K for K-Means
print("Testing K values from 2 to 10...")

k_range = range(2, 11)
silhouette_scores = []
davies_bouldin_scores = []

for k in k_range:
    print(f"Testing K={k}...", end=" ")
    
    # Fit K-Means
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X_scaled)
    
    # Calculate metrics
    sil_score = silhouette_score(X_scaled, labels)
    db_score = davies_bouldin_score(X_scaled, labels)
    
    silhouette_scores.append(sil_score)
    davies_bouldin_scores.append(db_score)
    
    print(f"Silhouette: {sil_score:.4f}, DBI: {db_score:.4f}")

# Find best K based on silhouette score
best_k_idx = np.argmax(silhouette_scores)
best_k = k_range[best_k_idx]
best_silhouette = silhouette_scores[best_k_idx]
best_dbi = davies_bouldin_scores[best_k_idx]

print(f"\nBest K selected: {best_k}")
print(f"Silhouette Score: {best_silhouette:.4f}")
print(f"Davies-Bouldin Index: {best_dbi:.4f}")


In [ ]:

# Train final K-Means model
print(f"Training final K-Means model with K={best_k}...")

kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=20)
kmeans_labels = kmeans_final.fit_predict(X_scaled)

# Analyze cluster distribution
unique_labels, counts = np.unique(kmeans_labels, return_counts=True)
cluster_distribution = dict(zip(unique_labels, counts))

print(f"K-Means model trained successfully!")
print(f"\nCluster distribution:")
for cluster_id, count in sorted(cluster_distribution.items()):
    percentage = count / len(kmeans_labels) * 100
    print(f"Cluster {cluster_id}: {count} days ({percentage:.1f}%)")

# Save K-Means model
with open(MODELS_DIR / "kmeans_model.pkl", "wb") as f:
    pickle.dump(kmeans_final, f)

print(f"K-Means model saved to: {MODELS_DIR / 'kmeans_model.pkl'}")


In [ ]:

# Train DBSCAN model
print("Training DBSCAN model...")

eps_start = 1.6
min_samples = 8
eps = eps_start
dbscan = None
dbscan_labels = None

for attempt in range(10):
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    dbscan_labels = dbscan.fit_predict(X_scaled)
    
    noise_count = np.sum(dbscan_labels == -1)
    cluster_count = len(np.unique(dbscan_labels[dbscan_labels != -1]))
    
    print(f"Attempt {attempt + 1}: eps={eps:.1f}, noise_points={noise_count}, clusters={cluster_count}")
    
    if not np.all(dbscan_labels == -1):  # Not all points are noise
        break
    eps += 0.4

# Analyze DBSCAN results
noise_count = np.sum(dbscan_labels == -1)
non_noise_labels = dbscan_labels[dbscan_labels != -1]
unique_dbscan_clusters = np.unique(non_noise_labels)

print(f"\nDBSCAN trained successfully!")
print(f"Final eps: {eps:.1f}")
print(f"Noise points: {noise_count} ({noise_count/len(dbscan_labels)*100:.1f}%)")
print(f"Number of clusters: {len(unique_dbscan_clusters)}")

# Save DBSCAN model
with open(MODELS_DIR / "dbscan_model.pkl", "wb") as f:
    pickle.dump(dbscan, f)

print(f"DBSCAN model saved to: {MODELS_DIR / 'dbscan_model.pkl'}")


In [ ]:

# Create clustering results dataframe
clustering_results = pd.DataFrame({
    'date': daily_profiles['date'],
    'kmeans_cluster': kmeans_labels,
    'dbscan_cluster': dbscan_labels,
    'dbscan_noise_flag': (dbscan_labels == -1).astype(int),
    'daily_mean': daily_profiles['daily_mean'],
    'daily_peak': daily_profiles['daily_peak'],
    'daily_std': daily_profiles['daily_std'],
    'day_of_week': daily_profiles['day_of_week'],
    'month': daily_profiles['month'],
    'is_weekend': daily_profiles['is_weekend']
})

# Save clustering results
clustering_results.to_csv(CLUSTERING_RESULTS_PATH, index=False)

print(f"Clustering results saved to: {CLUSTERING_RESULTS_PATH}")
print(f"Results shape: {clustering_results.shape}")


In [ ]:

# Calculate and save clustering metrics
clustering_metrics = {
    'best_k': int(best_k),
    'silhouette_score': round(float(best_silhouette), 4),
    'davies_bouldin_index': round(float(best_dbi), 4),
    'cluster_distribution': {int(k): int(v) for k, v in cluster_distribution.items()},
    'dbscan_noise_count': int(noise_count),
    'dbscan_cluster_count': int(len(unique_dbscan_clusters)),
    'dbscan_parameters': {
        'eps': round(float(eps), 2),
        'min_samples': min_samples
    },
    'data_info': {
        'total_days': len(daily_profiles),
        'features_used': len(available_cols),
        'date_range_start': daily_profiles['date'].min(),
        'date_range_end': daily_profiles['date'].max()
    }
}

# Save metrics
with open(MODELS_DIR / "clustering_metrics.json", 'w') as f:
    json.dump(clustering_metrics, f, indent=2)

print(f"Clustering metrics saved to: {MODELS_DIR / 'clustering_metrics.json'}")

print("\n=== CLUSTERING TRAINING SUMMARY ===")
print(f"Best K for K-Means: {best_k}")
print(f"Silhouette Score: {best_silhouette:.4f}")
print(f"Davies-Bouldin Index: {best_dbi:.4f}")
print(f"DBSCAN clusters: {len(unique_dbscan_clusters)}")
print(f"DBSCAN noise points: {noise_count}")
print(f"\nSaved files:")
print(f"- {MODELS_DIR / 'profile_scaler.pkl'}")
print(f"- {MODELS_DIR / 'kmeans_model.pkl'}")
print(f"- {MODELS_DIR / 'dbscan_model.pkl'}")
print(f"- {CLUSTERING_RESULTS_PATH}")
print(f"- {MODELS_DIR / 'clustering_metrics.json'}")
